In [1]:
import pyvisa
import matplotlib
import time
rm = pyvisa.ResourceManager()

#192.168.88.11 - TH1992B
#192.168.88.12 - TH2690A
DeviceAddress = '192.168.88.12'
DevicePort = '45454'
TCPIP = f'TCPIP0::{DeviceAddress}::{DevicePort}::SOCKET'

DeviceSerial = 'W152230156'
USB = '' #USB0::0x1105::0x1992::W152230156::INSTR
res = rm.list_resources()
for i in res:
    if DeviceSerial in i:
        USB = i

try:
    tonghui = rm.open_resource(TCPIP, read_termination='\n')
    #tonghui.timeout = 5000
    print(f'{tonghui.query("*IDN?")} - подключено по TCPIP')
except:
    try:
        tonghui = rm.open_resource(USB)
        print(f'{tonghui.query("*IDN?")} - подключено по USB')
    except:
        print('не подключено')


TH2690A,V1.0.13,R101230154 - подключено по TCPIP


In [13]:
rm = pyvisa.ResourceManager()
        
DeviceAddress = '192.168.88.11'
DevicePort = '45454'
TCPIP = f'TCPIP0::{DeviceAddress}::{DevicePort}::SOCKET'

DeviceSerial = 'W152230156'
#'ASRL5::INSTR'
#USB0::0x1105::0x1992::W152230156::INSTR
USB, USBtype = '', ''
res = rm.list_resources()
#print(res)
for i in res:
    if DeviceSerial in i:
        USB, USBtype = i, 'USB TCM'
    else:
        #предполагаем, что наш девайс - первый в списке
        USB, USBtype = res[0], 'USB CDC'

try:
    tonghui = rm.open_resource(TCPIP, read_termination='\n')
    print(f'{tonghui.query("*IDN?")} - подключено по TCPIP')
except:
    try:
        tonghui = rm.open_resource(USB, )
        print(f'{tonghui.query("*IDN?")} - подключено по {USBtype}')
        #self.tonghui.baud_rate = 115200
    except:
        print('не подключено')
else:
    if tonghui.query('*OPC?') == '1':
        print('Очередь команд свободна')
    else:
        print('Что-то не так с очередью команд')
        #тут можно попробовать это исправить
        #вроде как если измерить при помощи meas

Tonghui,TH1992B,Ver 1.6.7 - подключено по TCPIP
Что-то не так с очередью команд


In [12]:
# Check if pyvisa-py can see USB
rm = pyvisa.ResourceManager('@py')
#print(rm.list_backends())  # Should show 'usb' as available

print(res)

()


In [14]:
import serial
import serial.tools.list_ports

print(serial.tools.list_ports.comports())

# Find your device
for port in serial.tools.list_ports.comports():
    if 'Tonghui' in port.description or 'TH1992' in port.description:
        com_port = port.device
        print(f"Found device on {com_port}")

In [2]:
tonghui.close()

In [17]:
channel, state = 1, 1
enable_channel = f'OUTP{channel}:STAT {state}'
tonghui.write(enable_channel)
print(tonghui.query(f'OUTP{channel}:STAT?'))

1


In [49]:
tonghui.write(':SOUR1:FUNC:MODE?')
print(tonghui.read())

VOLT


In [11]:
tonghui.query(':MEAS?')

'Error 2--Need parameters'

In [158]:
command = '*TRG'
tonghui.write(command)


6

In [156]:
selected_format = 'VOLT,CURR,TIME'
tonghui.write(':FORM:ELEM:SENS ' + selected_format)

32

In [17]:
print(tonghui.query(':MEAS? (@1)'))

+5.000012E-04,+5.429119E-02


In [20]:
print(tonghui.query(':FETC?'))

NaN


In [3]:
print(tonghui.query('*OPC?'))

0


In [15]:
def read_data(device):
    data, byte = '', ''
    while True:
        try:
            byte = device.read_bytes(1)
            data += byte.decode()
        except:
            break
    return data

answer = read_data(tonghui)
print(answer, type(answer))

 <class 'str'>


In [48]:
#:INITiate[:IMMediate]<:ACQuire|:TRANsient|[:ALL]> [chanlist]
command = ':TRIG:SOUR BUS'
tonghui.write(command)

16

In [49]:
command = ':TRIG:IMM'
tonghui.write(command)

11

In [7]:
#[:SOURce[c]]:<CURRent|VOLTage>[:LEVel][:IMMediate][:AMPLitude]?
#command = ':SOUR1:VOLT:LEV:IMM:AMP?'

#[:SOURce[c]]:<CURRent|VOLTage>:MODE? (FIX, SWE, LIST)
#command = ':SOUR1:VOLT:MODE?'
#[:SOURce[c]]:<CURRent|VOLTage>:RANGe?
#the command is valid when the auto-set output range function is disabled
#see Table 2-6 for Voltage Source Output Range Table and Table 2-7 for Current Source Output Range Table
#command = 'SOUR:VOLT:RANG?'
#[:SOURce[c]]:<CURRent|VOLTage>:RANGe:AUTO? (1, 0)
#command = 'SOUR1:VOLT:RANG:AUTO?'
#command = 'SOUR1:VOLT:RANG?'

#:OUTPut[c]:HCAPacitance[:STATe] <mode> (ON, OFF)
command = ':OUTP1:HCAP:STAT?'

#:SENSe:<CURRent[:DC]|RESistance|VOLTage[:DC]>:RANGe:A UTO
#command = ':SENS1:CURR:RANG:AUTO?'

#:SENSe:<CURRent[:DC]|RESistance|VOLTage[:DC]>:RANGe[: UPPer]
#command = ':SENS:CURR:RANG:UPP?'

#:OUTPut[c][:STATe]?
#command = 'OUTP1:STAT?'

#command = ':SOUR1:VOLT?'

#:SENSe[c]:<CURRent[:DC]|VOLTage[:DC]>:PROTection[:LEVel] <compliance>
#command = ':SENS1:CURR:PROT:LEV?'

#:OUTPut[c]:LOW <low_state>
#command = ':OUTP1:LOW?'

#:INITiate[:IMMediate]<:ACQuire|:TRANsient|[:ALL]> [chanlist]
#command = ':INIT:ACQ @1'

#command = '*TRG'
#a = tonghui.write(command)


#tonghui.write(command[:-1] + ' 1')
#time.sleep(0.2)

#command = ':SOUR1:VOLT?'
command = ':SYST:HAND:TIM?'
command = ':TRIG:HELP?'

#command = 'OPC?'
ans = tonghui.query(command)
print(ans)

Error 5--Not found


In [16]:
tonghui.write('SOUR1:CURR:MODE SWE')

21

In [49]:
channel, val = 1, '?'
#tonghui.query(f':SOUR{channel}:FUNC:MODE?')
#:OUTPut[c]:FILTer[:LPASs]:TCONstant <time_constant>
#:OUTPut[c]:FILTer[:LPASs][:STATe]?
command=':OUTP{ch}:FILT{val}'.format(ch=channel, val=val)
ans = tonghui.query(command)
print(ans)

1


In [8]:
tonghui.write(':TRIG1:ACQ:COUN 500')
print(tonghui.query(':TRIG1:ACQ:COUN?'))
a = tonghui.write(':INIT:IMM:ACQ (@1)')
print(a)

500
20


In [103]:
tonghui.write(':DISP:DIG 7')

13

In [7]:
print(tonghui.query(':DISP:DIG?'))

7


In [10]:
print(tonghui.query(':SOUR1:CURR?'))

0.0005


In [109]:
print(tonghui.write(':DISP:DIG?'))
print(tonghui.read())

12
7


In [95]:
#TRIGger[c]<:ACQuire|:TRANsient|[:ALL]>:DELay?
src_trig_delay, meas_trig_delay = tonghui.query(':TRIG1:TRAN:DEL?'), tonghui.query(':TRIG1:ACQ:DEL?')
print(f'Src Trig Delay  \t= {src_trig_delay} sec\nMeas Trig Delay \t= {meas_trig_delay} sec')

Src Trig Delay  	= Error 5--Not found sec
Meas Trig Delay 	= +5.251778E-11,+4.815998E-07 sec


In [42]:
#TRIGger[c]<:ACQuire|:TRANsient|[:ALL]>:COUNt?
src_trig_count, meas_trig_count = tonghui.query(':TRIG1:TRAN:COUN?'), tonghui.query(':TRIG1:ACQ:COUN?')
print(f'Src Trig Count  \t= {src_trig_count}\nMeas Trig Count \t= {meas_trig_count}')

Src Trig Count  	= 201
Meas Trig Count 	= 201


In [13]:
#TRIGger[c]<:ACQuire|:TRANsient|[:ALL]>:TIMer?
src_trig_period, meas_trig_period = tonghui.query(':TRIG1:TRAN:TIM?'), tonghui.query(':TRIG1:ACQ:TIM?')
print(f'Src Trig Period \t= {src_trig_period} sec\nMeas Trig Period\t= {meas_trig_period} sec')

Src Trig Period 	= 0.02 sec
Meas Trig Period	= 0.5 sec


In [21]:
tonghui.write(':SENS1:CURR:RANG:AUTO ON')

26

In [8]:
#[:SOURce]:<CURRent|VOLTage>:<STARt|STOP>
#[SOURce]:<CURRent|VOLTage>:STEP
#[:SOURce[c]]:SWEep:POINts?

ans = tonghui.query(':SOUR1:VOLT:STOP?')
print(ans)

1


In [2]:
print(tonghui.query(':SENS1:DATA? STAR,5')) # <offset,size>

-8.014317E+00,-4.686953E-07,-9.900166E+00,-1.008027E-07,-9.800169E+00,-9.991284E-08,-9.700179E+00,-9.882395E-08,-9.600163E+00,-9.774641E-08


In [6]:
print(tonghui.query(':SENS1:DATA:LAT?'))

+1.109954E-07,+9.999819E+00


In [2]:
#print(tonghui.query(':FETCH:ARR? (@1,2)'))

In [3]:
print(tonghui.query(':SOUR1:FUNC:MODE?'))

protlevel = ':SENS1:VOLT:PROT:LEV?'
print(tonghui.query(f':SENS1:VOLT:PROT:LEV?'))

#V-range
#The mode is 0 or OFF, indicating that the auto-set measurement range function is disabled,
#and the measurement is performed with the range set by
#:SENSe:<CURRent[:DC]|RESistance|VOLTage[:DC]>: RANGe[:UPPer] command.
#The mode is 1 or ON, indicating that the auto-set measurement range function is enabled, 
#and will automatically set the range that provides the best resolution for the measurement.
vrange_auto = ':SENS1:VOLT:RANG:AUTO 1' #1, 0, ':SENS1:VOLT:RANG:AUTO?'
vrange_automode = ':SENS1:VOLT:RANG:AUTO:MODE?' #NORMal, RESolution, SPEed - мануал стр.99
#This command is to set the threshold value of auto-set measurement range operation.
range_auto_thresh = ':SENS1:VOLT:RANG:AUTO:THR?'

tonghui.write(':SENS1:VOLT:RANG:UPP 2')
print(tonghui.query(':SENS1:VOLT:RANG:UPP?'))
print(tonghui.query(':SENS2:VOLT:RANG:UPP?'))

tonghui.write(vrange_auto)
print(tonghui.query(':SENS1:VOLT:RANG:AUTO?'))
print(tonghui.query(vrange_automode))
#print(tonghui.query(range_auto_thresh))

print(tonghui.query(':SENS1:CURR:RANG:AUTO?')) #только в режиме VOLT, и наоборот

CURR
2
2
2
ON
SPEed
Source mode error


In [6]:
tonghui.write(':SOUR1:FUNC:MODE CURR')
print(tonghui.query(':SOUR1:FUNC:MODE?'))
print(tonghui.query(':SENS1:CURR:RANG:AUTO?'))



23
CURR
Source mode error


In [19]:
print(tonghui.write(":SYST:COMM:USB:TYP TMC"))

24


In [7]:
from datetime import datetime
import time

#formatted_date = now.strftime("%Y-%m-%d %H:%M:%S")
#t1 = datetime.time()
t2 = datetime.now().strftime("%H:%M:%S:%f")

tt1 = time.time()
time.sleep(1)
tt2 = time.time()-tt1
#500, 1500, 3000, 4500
print(f'{datetime.now()}'[11:19])

14:16:35


In [36]:
print(tonghui.query('FETCH:ALL?'))

0.000000E+00,1.898634E-05,0.000000E+00,2025-08-07 19:24:51,2.000000E+01,1.898634E-05,9.9900E+02,9.9900E+02


In [49]:
#TH2690A

#MeasSet
#Function 'FUNC:FUNC RES|VOLT|CURR|SRC'
tonghui.write('FUNC:FUNC CURR')
#Range 'RES|VOLT|CURR:RANGE <>'
tonghui.write('CURR:RANGE 5')
#Speed 'RES|VOLT|CURR:SPEED FAST|MID|SLOW'
tonghui.write('CURR:SPEED MID')

#Source switch 'FUNC:SRC ON|OFF'
tonghui.write('FUNC:SRC ON')
#Ammeter switch 'FUNC:AMMET ON|OFF'
tonghui.write('FUNC:AMMET ON')
#Run/Stop measurement 'FUNC:RUN|STOP'
time.sleep(1) #если не подождать, то измерение стартанет, но кнопка не загорится
tonghui.write('FUNC:RUN')

tonghui.write('DISP:PAGE MEAS')



16

In [92]:
tonghui.write('FUNC:RUN')

10

In [18]:
tonghui.write('SRC:VALUE -2')

14

In [17]:
tonghui.write('SRC:RANGE 1')

13

In [15]:
tonghui.read()

VisaIOError: VI_ERROR_TMO (-1073807339): Timeout expired before operation completed.

In [6]:
all_results = tonghui.query('FETCH:ALL?').split(',')
        
results = {'volt':all_results[0],
           'curr':all_results[1],
           'char':all_results[2],
           'time':all_results[3],
           'vsource':all_results[4],
           'math':all_results[5],
           'temp':all_results[6],
           'hum':all_results[7],
          }

print(results['curr'])

1.500000E-10


In [16]:
#print(tonghui.query('FUNC:FUNC?'))

def read_data(device):
    data, byte = '', ''
    while True:
        try:
            byte = device.read_bytes(1)
            data += byte.decode()
        except:
            break
    return data

def custom_query(device, command):
    device.write(command)
    return read_data(device)
    
#print(custom_query(tonghui, 'FUNC:FUNC?', ))
print(custom_query(tonghui, 'SRC:RANGE?', ))
#print(read_data(tonghui))

0~1000V


In [36]:
def test(t):
    return t, (time.sleep(3) if t == 2 else None)

print(test(3))

(3, None)


In [ ]:
ppd = True
if ppd, time.sleep(3):
    

In [1]:
28.74+26.29

55.03